# Experiment: does scoring scenes in parallel change any number? (CPU, high RAM)

Scores the development block again under a separate run name, with scenes scored in parallel processes, and compares
every number with the earlier one-at-a-time run. It must print IDENTICAL. It also shows how much faster it was.

**Outcome: identical, 4.8 times faster on 8 cores.** Built into `pipeline.run_jobs`, which `03_score` uses.

In [1]:
# --- 1. Configuration ---
PERSIST_MODE = "drive"
CAMERA = "front_medium"
MODELS = ["vggt_omega_512", "vggt_1b"]
SPLIT, BLOCK = "val", 11
REFERENCE_RUN = "phase7_front_medium"      # scored one scene at a time; that run matched `05b_one_block_metrics` exactly
CHECK_RUN = "parallel_check"
WORKERS = None                             # None = one per CPU core, at most 8

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Start the session ---
import os
from vggt_aura.session import start_session

# build_cpp=True compiles the C++ geometry core on this server (about 15 s). Ground truth is then built
# with it, which gives exactly the same result as the Python reference, faster. If the build fails,
# everything still runs, in Python.
session = start_session(persist_mode=PERSIST_MODE, require_gpu=False, build_cpp=True)
print("CPU cores on this server:", os.cpu_count())

Mounted at /content/drive
installing vggt_omega
installing pybind11
installing fzi_aura
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)
CPU cores on this server: 8


In [4]:
# --- 4. Score the block again, in parallel, under its own run name ---
import shutil
import pandas as pd
from vggt_aura import aura_data as ad, pipeline as pl

chunks, scene_blocks, hub_files = ad.fetch_release_tables(session.data_root / "_release_tables")
EXCLUDED = ad.fetch_excluded_scene_ids(session.data_root / "_release_tables")
shutil.rmtree(session.persist_root / "metrics" / CHECK_RUN, ignore_errors=True)     # always a fresh comparison
summary = pl.process_block(session, SPLIT, BLOCK, ad.block_scene_ids(scene_blocks, SPLIT, BLOCK, EXCLUDED), CAMERA, MODELS,
                           CHECK_RUN, scene_names=ad.block_scene_names(scene_blocks, SPLIT, BLOCK, EXCLUDED), workers=WORKERS)
print(summary)

  predictions and ground truth are all saved: fetching the camera layer only
$ /usr/bin/python3 -m fzi_aura.download /content/data/fzi-aura/val_block000011 --revision 3404bd6b8fcd6eed53a0ec7610650a6393aabb49 --splits val --scene-ids-file /content/data/fzi-aura/val_block000011/_scene_ids_val_block000011.txt --layers camera_keyframes --jobs 8 --verify
  predictions: 0 made now, the rest loaded (137 s) | scoring 20 scenes with 8 worker(s)
  2025-07-15-13-11-47|24      86.8 s | vggt_omega_512: truth loaded | vggt_1b: truth loaded
  2025-06-11-13-47-25|25      66.1 s | vggt_omega_512: truth loaded | vggt_1b: truth loaded
  2025-06-13-07-09-37|78      68.0 s | vggt_omega_512: truth loaded | vggt_1b: truth loaded
  2025-07-15-14-15-47|71      80.5 s | vggt_omega_512: truth loaded | vggt_1b: truth loaded
  2025-07-15-13-11-47|63      82.9 s | vggt_omega_512: truth loaded | vggt_1b: truth loaded
  2025-06-11-12-27-55|207     54.0 s | vggt_omega_512: truth loaded | vggt_1b: truth loaded
  2025-0

In [5]:
# --- 5. Compare every number with the reference run ---
all_identical = True
for model in MODELS:
    ref_rows, ref_scenes = pl.load_run(session.persist_root, REFERENCE_RUN, model)
    new_rows, new_scenes = pl.load_run(session.persist_root, CHECK_RUN, model)
    keys = ["scene_id", "protocol", "stratum_type", "stratum"]
    a = ref_rows.sort_values(keys).reset_index(drop=True)
    b = new_rows.sort_values(keys).reset_index(drop=True)
    same_rows = a.equals(b)
    numeric = [c for c in ref_scenes.columns if pd.api.types.is_numeric_dtype(ref_scenes[c]) and c in new_scenes]
    sa = ref_scenes.sort_values("scene_id").reset_index(drop=True)[numeric]
    sb = new_scenes.sort_values("scene_id").reset_index(drop=True)[numeric]
    same_scenes = sa.equals(sb)
    all_identical &= same_rows and same_scenes
    print(f"{model}: {len(a)} score rows identical: {same_rows} | {len(sa)} scene records identical: {same_scenes}")
    if not same_rows:
        worst = (a["abs_rel"] - b["abs_rel"]).abs().max()
        print("   largest AbsRel difference:", worst)
print()
print("IDENTICAL: parallel scoring changes no number." if all_identical else "DIFFERENT: do not use parallel scoring, and report it.")
print(f"scoring took {summary.get('score_s')} s with {summary.get('workers')} worker(s); one at a time it took about 1040 s")

vggt_omega_512: 1940 score rows identical: True | 20 scene records identical: True
vggt_1b: 1935 score rows identical: True | 20 scene records identical: True

IDENTICAL: parallel scoring changes no number.
scoring took 216.5 s with 8 worker(s); one at a time it took about 1040 s
